## Loading the file containing all the tracks.

In [1]:
import json

with open('dataset/tracks_lyrics_p.json', 'r', encoding="utf-8") as json_file:
    artists = json.load(json_file)
    
print(artists[0]["artist_id"], "\n", 
      artists[0]["artist_name"], "\n", 
      artists[0]['tracks'][0]['title'], "\n", 
      artists[0]['tracks'][0]['lyrics'][:100].lstrip(), "...", 
      sep="")

582KhTHEVOONNQLmQ5612r
Calcutta 
Tutti
Ho messo le scarpe nuove per i giorni di fango
Forse i leghisti lì in riva al Po non hanno più un ca...


## Importing lingua and removing non italian text

As the first step in our preprocessing pipeline, we utilize the Python library lingua to filter out any non-Italian text from the lyrics. Additionally, we remove any song lyrics containing fewer than 8 lines, as a standard paragraph in Italian typically consists of about 12 lines.

The lingua Python library is designed for high-accuracy language detection on short texts. By supplying a predefined set of candidate languages, lingua calculates the probability that a given text belongs to each candidate and selects the most likely match. In our preprocessing pipeline, we evaluate the text line by line; any line where Italian is not identified as the winning language is immediately removed.

In [2]:
from lingua import Language, LanguageDetectorBuilder

# Define the expected languages to give the detector context.
# This is crucial to allow the algorithm to discard non-Italian lines.
expected_languages = [
    Language.ITALIAN, 
    Language.ENGLISH, 
    Language.SPANISH, 
    Language.FRENCH, 
    Language.RUSSIAN, 
    Language.GERMAN,
    Language.PORTUGUESE,
]
detector = LanguageDetectorBuilder.from_languages(*expected_languages).build()

def filter_foreign_lines(lyrics):
    """
    Splits the lyrics into lines, detects the language of each line,
    and retains only the lines identified as Italian.
    """
    italian_lines = []
    
    # Split the lyrics by newline character
    lines = lyrics.split('\n')
    
    for line in lines:
        stripped_line = line.strip()
        
        if not stripped_line:
            continue
            
        # Detect the language of the current line
        detected_language = detector.detect_language_of(stripped_line)
        
        if detected_language == Language.ITALIAN:
            italian_lines.append(stripped_line)
        else:
            # print(f"Removed line: {stripped_line} (Language: {detected_language})")
            pass
            
    return '\n'.join(italian_lines)


# Filter tracks by language first, then by valid length
removed_tracks_count = 0
valid_tracks_count = 0
for artist in artists:
    valid_tracks = [] # List to store tracks that survive both filters
    
    for track in artist.get('tracks', []):
        raw_lyrics = track.get('lyrics', '')
        
        # Language Filter
        # Extract only the Italian lines from the lyrics
        filtered_lyrics = filter_foreign_lines(raw_lyrics)
        
        # Length Filter (Applied to the remaining Italian text)
        # Split the filtered text by newline and count the valid lines
        # We use .strip() to ignore completely empty lines or lines with just spaces
        valid_italian_lines_count = len([line for line in filtered_lyrics.split('\n') if line.strip()])
        
        # If the surviving Italian text has 8 or more lines, we keep the track
        if valid_italian_lines_count >= 8:
            track['lyrics'] = filtered_lyrics
            valid_tracks.append(track)
            valid_tracks_count += 1
        else:
            # The track is either too short overall, or too much foreign text was removed
            removed_tracks_count += 1
            
    artist['tracks'] = valid_tracks
    
print(f"Total tracks removed due to insufficient Italian content: {removed_tracks_count}")
print(f"Total valid tracks remaining after filtering: {valid_tracks_count}")

Total tracks removed due to insufficient Italian content: 6571
Total valid tracks remaining after filtering: 20616


## Text pre-processing

### spaCy

First, we cleaned the text by replacing all newline characters with spaces and normalizing it using regular expressions. We preserved the accents, as they are essential for distinguishing between different words in the Italian language. For tokenization and lemmatization, we initially turned to spaCy, a library widely regarded as the industry standard. Given the choice between their small and large Italian language packages, we opted for the large model. _(to install it: python -m spacy download it_core_news_lg)_

Because processing the lyrics sequentially proved to be too slow, we optimized our pipeline using multiprocessing.
However, upon evaluating the output on our first track **('Tutti' by Calcutta)**, we discovered significant inaccuracies in the lemmatization process. For instance, the conjugated verb **'vesto'** was incorrectly lemmatized to **'vestare'** a non-existent word in Italian. Due to these critical errors, we abandoned spaCy in favor of another library called Stanza, which is detailed below.

In [ ]:
import spacy
import re
import time

# Load the Italian NLP model
nlp = spacy.load("it_core_news_lg")

def regex_clean(text):
    """
    Applies extremely fast regular expression normalizations.
    It is much more efficient to do this BEFORE passing the text to spaCy.
    """
    text = text.replace('\n', ' ')
    text = re.sub(r'[^a-zA-Zàèéìíòóùú]', ' ', text).lower()
    return re.sub(r'\s+', ' ', text).strip()


# PREPARATION PHASE
# Flatten the nested dictionary structure to process lyrics in bulk
flat_tracks = []
raw_texts = []


# Collect all valid texts and their corresponding track objects
for artist in artists:
    for track in artist.get('tracks', []):
        flat_tracks.append(track)
        raw_texts.append(regex_clean(track.get('lyrics', '')))


# PARALLEL PROCESSING PHASE
print(f"Initiating parallel processing for {len(raw_texts)} tracks...")

corpus = []

# The Magic of nlp.pipe():
# as_tuples=False: We only pass the texts
# n_process=-1: Tells spaCy to use ALL available CPU cores. 
# (If it crashes due to OS limits, change to a fixed number like 2 or 4)
# batch_size=100: Sends 100 songs to each core at a time, minimizing memory overhead
doc_generator = nlp.pipe(
    raw_texts, 
    n_process=-1, 
    batch_size=100, 
)

# Zip binds the processed 'doc' back to its original 'track' dictionary
for track, doc in zip(flat_tracks, doc_generator):
    lemmas = []
    for token in doc:
        
        # Exclude extremely short tokens
        if len(token.text) >= 2:
            lemmas.append(token.lemma_)
            
    # Reconstruct the string
    clean_lyrics = " ".join(lemmas)
    
    # Save back to the original nested dictionary structure
    track['lemmatized_lyrics'] = clean_lyrics
    
    # Add to our flat corpus list for Scikit-Learn Vectorization later
    corpus.append(clean_lyrics)

print(f"Parallel preprocessing completed")

### NLP Pipeline with Stanza (Stanford NLP)

**Stanza** is a powerful NLP library developed by Stanford University that relies on deep neural networks. It provides excellent native support for the Italian language and is explicitly designed to leverage GPU acceleration via PyTorch.

Stanza operates on a **pipeline architecture**. A neural network cannot extract a word's root directly from raw text in a single pass; the process must be executed sequentially. Each tool in the pipeline prepares the data structure for the subsequent one. This is why we initialize our model with these three specific components: `stanza.Pipeline(lang='it', processors='tokenize,pos,lemma', use_gpu=True)`.

Before feeding the text into Stanza, we applied a text normalization step, specifically ensuring that Italian accents were preserved to maintain full semantic integrity.

Here is a breakdown of our pipeline processors:

* **`tokenize` (Tokenization):** This step takes the raw text and slices it. It first performs *Sentence Segmentation* (dividing the text into distinct sentences) and then *Tokenization* (splitting each sentence into individual words and punctuation marks).
* **`pos` (Part-Of-Speech Tagging):** It examines each token within the context of the entire sentence and assigns a grammatical label (e.g., Noun, Verb, Adjective, Adverb, Pronoun). This contextual understanding drastically improves accuracy compared to standard spaCy models. Furthermore, having defined the grammatical class of each element, we can implement an advanced filtering step: discarding structural "stop-words" (like articles and prepositions) that add no value to our upcoming Bag-of-Words (BoW) and TF-IDF matrices.
* **`lemma` (Lemmatization):** It takes the original token alongside its POS tag to accurately compute the dictionary root of the word (the lemma). Extracting the pure lemmas is our ultimate goal for cleaning the BoW and TF-IDF matrices. Thanks to the foundational work done by the POS tagger, the lemmatizer resolves contextual ambiguities and operates flawlessly.

#### Performance Note
It is important to highlight that, despite utilizing GPU acceleration, Stanza's deep learning approach is significantly slower than spaCy. On our specific hardware configuration (**CPU: Intel Ultra 7 265k, RAM: 32GB DDR5 6400MHz, GPU: Nvidia RTX 5070FE**), the entire text processing phase took approximately **3 hours** to complete.

**Bibliographic reference (Stanza):**
> Qi, P., Zhang, Y., Zhang, Y., Bolton, J., & Manning, C. D. (2020). **Stanza: A Python Natural Language Processing Toolkit for Many Human Languages**. In *Proceedings of the 58th Annual Meeting of the Association for Computational Linguistics: System Demonstrations* (pp. 101-108).

In [3]:
import stanza
import re
import time

# INITIALIZATION
# Download the Italian neural model (only needs to be run once)
# stanza.download('it')

# Initialize the pipeline
# use_gpu=True automatically sends the tensors to your NVIDIA card
# processors='tokenize,pos,lemma' loads ONLY the tools we need to save RAM
nlp_stanza = stanza.Pipeline(lang='it', processors='tokenize,pos,lemma', use_gpu=True)

def regex_clean(text):
    """
    Applies fast regular expression normalizations before NLP.
    """
    text = text.replace('\n', ' ')
    text = re.sub(r'[^a-zA-Zàèéìíòóùú]', ' ', text).lower()
    return re.sub(r'\s+', ' ', text).strip()


# PREPARATION PHASE
flat_tracks = []
raw_texts = []

for artist in artists:
    for track in artist.get('tracks', []):
        flat_tracks.append(track)
        raw_texts.append(regex_clean(track.get('lyrics', '')))


# GPU NEURAL PROCESSING PHASE
print(f"Initiating Stanza Neural processing for {len(raw_texts)} tracks...")
start_time = time.time()

corpus = []

# Define the set of valid POS tags ONCE
VALID_POS_TAGS = {'NOUN', 'VERB', 'ADJ', 'ADV'}

# Process each track. Stanza automatically handles GPU batching internally
# for document objects, though it processes them sequentially in the loop.
for track, text in zip(flat_tracks, raw_texts):
    
    # Pass the text to the neural network
    doc = nlp_stanza(text)
    
    lemmas = []
    
    # Stanza structures text into sentences, then words
    for sentence in doc.sentences:
        for word in sentence.words:
            
            # Filter 1: Check the Part-Of-Speech (POS) tag against our pre-defined set. 
            # We ONLY keep Nouns (NOUN), Verbs (VERB), Adjectives (ADJ), and Adverbs (ADV)
            # This automatically acts as an incredibly powerful, grammar-based stop-word remover
            if word.upos in VALID_POS_TAGS:
                # Filter 2: extremely short tokens
                if len(word.text) >= 2:
                    # Stanza stores the base form in word.lemma
                    lemmas.append(word.lemma)
                
    clean_lyrics = " ".join(lemmas)
    track['lemmatized_lyrics'] = clean_lyrics
    corpus.append(clean_lyrics)

end_time = time.time()
print(f"Neural processing completed in {end_time - start_time:.2f} seconds.")

/home/gabriele11231/.venvs/rapids25.06_python3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-06-08 23:20:24 INFO: Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES
2026-06-08 23:20:25 INFO: Downloaded file to /home/gabriele11231/.cache/stanza/1.12.0/resources/resources.json
2026-06-08 23:20:25 WARNING: Language it package default expects mwt, which has been added
2026-06-08 23:20:25 INFO: Loading these models for language: it (Italian):
| Processor | Package           |
---------------------------------
| tokenize  | combined_nocharlm |
| mwt       | combined          |
| pos       | combined_charlm   |
| lemma     | combined_nocharlm |

2026-06-08 23:20:25 

Initiating Stanza Neural processing for 20616 tracks...
Neural processing completed in 9375.42 seconds.


As we can see below, using againg the song 'Tutti' by Calcutta as an example, the word vesto is now lemmatized as vestire.

In [4]:
def highlight(text, word):

    match = re.search(fr'((?:\S+\s+){{0,3}})\b({word})\b((?:\s+\S+){{0,3}})', text, re.IGNORECASE)
    
    p_before, target, p_after = match.groups(default="")
    return f"{p_before}\033[91m{target}\033[0m{p_after}".strip().replace('\n', ' ')

print("Original Lyrics:\n", highlight(artists[0]['tracks'][0]['lyrics'], "vesto"))
print("\nLemmatized Lyrics:\n", highlight(artists[0]['tracks'][0]['lemmatized_lyrics'], "vestire"))

Original Lyrics:
 ed io mi vesto di bianco Vorrei

Lemmatized Lyrics:
 più capobranco rivoluzione vestire bianco tenere mano


In [ ]:
with open("dataset/tracks_lyrics_preprocessed.json", 'w', encoding="utf-8") as json_file:
        json.dump(artists, json_file, ensure_ascii=False, indent=4)

### Feature Extraction: BoW and TF-IDF

**scikit-learn** is an industry-standard machine learning library for Python. Once our text has been cleaned, tokenized, and lemmatized by Stanza, it remains in a string format. However, Machine Learning algorithms cannot process raw text; they require numerical inputs. To bridge this gap, we rely on scikit-learn to perform **vectorization**, converting our linguistic corpus into structured mathematical matrices.

Before generating these matrices, we must define **statistical thresholds** to filter out noise. Even after discarding structural stop-words during the Stanza pipeline, a corpus often contains domain-specific clutter or rare anomalies that add computational weight without predictive value. We handle this using document frequency boundaries:

* **`max_df=0.75` (Maximum Document Frequency):** this threshold aggressively removes terms that appear in more than 75% of the tracks. In a musical context, for example, overly ubiquitous words like 'amore' or 'cuore' lose their discriminative power. Because they appear almost everywhere, they act as domain-specific stop-words and are discarded.
* **`min_df=0.01` (Minimum Document Frequency):** conversely, this discards words that appear in fewer than 1% of the tracks. This step effectively purges our matrices of rare typos, highly idiosyncratic slang, or single-use terms that would only inflate the matrix's dimensionality without improving the model's accuracy.

Here is a breakdown of our vectorization process:

* **`CountVectorizer` (Bag-of-Words):** this tool scans the processed corpus and tallies the absolute frequency of each retained lemma. The result is a high-dimensional Bag-of-Words (BoW) matrix where rows represent individual documents (tracks) and columns represent the discrete frequency of our filtered vocabulary. 
* **`TfidfVectorizer` (TF-IDF):** while BoW counts absolute occurrences, Term Frequency-Inverse Document Frequency (TF-IDF) weighs them. It diminishes the weight of terms that appear frequently across the entire corpus while boosting the weight of terms that are highly specific to individual documents. This highlights the unique thematic footprint of each track.

#### Methodological Note
It is crucial to highlight a specific architectural choice in our code: we initialize the `TfidfVectorizer` using `vocabulary=bow_vectorizer.vocabulary_`. 
By forcing the TF-IDF matrix to inherit the exact same feature space learned by the BoW vectorizer, we guarantee that both matrices possess identical dimensions and identical columns. This strict consistency is a best practice, ensuring seamless interoperability and preventing dimensional mismatches when feeding these matrices into downstream Machine Learning models for comparative analysis.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# STATISTICAL THRESHOLDS
# max_df = 0.75: Discard words appearing in > 50% of tracks (Domain-specific stop-words are too common to be informative)
# min_df = 0.01: Discard words appearing in < 1% of tracks (Rare typos, extremely specific slang)
MAX_DF_THRESHOLD = 0.75
MIN_DF_THRESHOLD = 0.01

print("--- 1. BAG OF WORDS (BoW) GENERATION ---")
# Initialize the BoW vectorizer
bow_vectorizer = CountVectorizer(max_df=MAX_DF_THRESHOLD, min_df=MIN_DF_THRESHOLD)
# Fit and transform the Stanza-processed corpus
bow_matrix = bow_vectorizer.fit_transform(corpus)
print(f"BoW Matrix shape: {bow_matrix.shape} (Documents, Features)")


print("\n--- 2. TF-IDF GENERATION ---")
# We initialize the TF-IDF matrix using the vocabulary learned by the BoW vectorizer.
tfidf_vectorizer = TfidfVectorizer(vocabulary=bow_vectorizer.vocabulary_)
# Transform the corpus into weighted TF-IDF values
tfidf_matrix = tfidf_vectorizer.fit_transform(corpus)
# Extract the final useful vocabulary to verify
useful_features = tfidf_vectorizer.get_feature_names_out()

print(f"TF-IDF Matrix shape: {tfidf_matrix.shape} (Documents, Features)")
# Check
print(f"\nSample of useful features retained: {list(useful_features)[100:110]}")

--- 1. BAG OF WORDS (BoW) GENERATION ---
BoW Matrix shape: (20616, 1191) (Documents, Features)

--- 2. TF-IDF GENERATION ---
TF-IDF Matrix shape: (20616, 1191) (Documents, Features)

Sample of useful features retained: ['basso', 'bastardo', 'bastare', 'battere', 'battito', 'beat', 'bellezza', 'bello', 'bene', 'bere']


### High-Performance Topic Model Evaluation: Grid Search and Custom Metrics

Finding the optimal number of topics ($K$) in Machine Learning is rarely an exact science; it requires empirical testing. To determine the most effective mathematical approach for our specific corpus, we implement a **Grid Search** evaluating three distinct dimensionality reduction algorithms—LDA, NMF, and LSA—across a range of $K$ values (from 3 to 10). 

Because unsupervised learning lacks a definitive "ground truth" to measure against, we cannot rely on standard accuracy metrics. Instead, we must evaluate the quality of the generated topics using two custom natural language metrics:

* **Topic Diversity:** A model might achieve high coherence by generating multiple identical topics, which provides no analytical value. Diversity counters this by calculating the proportion of unique words across all top words extracted by the model. 
  $$\text{Diversity} = \frac{|\text{Unique Words}|}{\text{Total Words extracted}}$$
  A score closer to **1.0** indicates that the model successfully separated distinct concepts without overlapping vocabularies.

* **UMass Topic Coherence:** This metric measures the semantic quality of a topic by checking how often its top words actually co-occur in the same documents. It operates under the assumption that words belonging to a coherent concept will frequently appear together in the original text. The algorithm computes the co-document frequency $D(w_i, w_j)$ of word pairs, applying smoothing to prevent mathematical errors. The formula is expressed as:
  $$C_{UMass} = \sum_{i>j} \log \left( \frac{D(w_i, w_j) + 1}{D(w_j)} \right)$$
  Because it calculates a logarithmic penalty for words that rarely co-occur, the resulting score is strictly negative. A score **closer to 0** indicates higher semantic coherence.

Here is a breakdown of the algorithms evaluated in our parallelized pipeline:

* **`LDA` (Latent Dirichlet Allocation):** a generative probabilistic model. It fundamentally assumes that documents are random mixtures over latent topics, and topics are distributions over words. Because it relies on discrete probability distributions, **LDA must be fed the absolute integers of the BoW matrix**, not the continuous weights of TF-IDF.
* **`NMF` (Non-Negative Matrix Factorization):** a linear algebraic model that factors high-dimensional vectors into a lower-dimensional space, strictly enforcing positive values. This additive nature makes the topics highly interpretable. NMF excels when fed the weighted, continuous values of our **TF-IDF matrix**.
* **`LSA` (Latent Semantic Analysis / TruncatedSVD):** A classic algebraic dimensionality reduction technique. While it does not enforce non-negativity (meaning terms can have negative associations with a topic), it is highly efficient at capturing variance. Like NMF, it utilizes the **TF-IDF matrix**.

#### Architectural Note
Topic modeling is computationally expensive. Running three algorithms across eight different $K$ configurations sequentially would be incredibly slow. To solve this, we leverage the **`joblib`** library. By wrapping our evaluation function in `Parallel(n_jobs=-1)`, we command Python to bypass the Global Interpreter Lock (GIL) and distribute the training tasks concurrently across 100% of the available CPU cores. Furthermore, we pre-calculate a binary boolean mask of our BoW matrix before launching the parallel workers; this drastically accelerates the matrix dot-products required for the UMass Coherence calculations.

Finally, to make sense of this multidimensional evaluation and objectively determine the best model using intrinsic metrics, we pass the results to **Plotly**. This generates an interactive, colorblind-friendly dashboard that allows us to visually balance the trade-off between Coherence and Diversity to make our final selection.

**Bibliographic reference (UMass Coherence):**
> Mimno, D., Wallach, H., Talley, E., Leenders, M., & McCallum, A. (2011). **Optimizing semantic coherence in topic models**. In *Proceedings of the 2011 conference on empirical methods in natural language processing* (pp. 262-272).

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from joblib import Parallel, delayed
from sklearn.decomposition import LatentDirichletAllocation, NMF, TruncatedSVD

print("Initializing High-Performance Parallel Topic Model Evaluator...")

# CONFIGURATION
# Define the range of 'K' (number of topics) to test
TOPIC_RANGE = [3, 4, 5, 6, 7, 8, 9, 10]
ALGORITHMS = ['LDA', 'NMF', 'LSA']
NO_TOP_WORDS = 10 # Number of words to evaluate for Coherence and Diversity

# Pre-calculate a binary Document-Term matrix. 
binary_bow_matrix = (bow_matrix > 0).astype(int)

# CUSTOM EVALUATION METRICS 
def calculate_diversity(topic_words_list):
    """
    Calculates Topic Diversity.
    Formula: (Number of unique words across all topics) / (Total words extracted)
    Range: 0.0 to 1.0 (Higher is better)
    """
    all_words = []
    for words in topic_words_list:
        all_words.extend(words)
    unique_words = set(all_words)
    return len(unique_words) / len(all_words)

def calculate_umass_coherence(topic_words_indices, binary_matrix):
    """
    Calculates UMass Coherence directly from the binary Bag of Words matrix.
    It checks how often the top words co-occur in the actual lyrics.
    (Closer to 0 is better, highly negative means poor coherence).
    """
    coherence_score = 0.0
    num_topics = len(topic_words_indices)
    
    for word_indices in topic_words_indices:
        topic_score = 0.0
        # Iterate over all word pairs (w_i, w_j) in the top words
        for i in range(1, len(word_indices)):
            for j in range(i):
                w_i = word_indices[i]
                w_j = word_indices[j]
                
                # Document frequency of w_j
                doc_freq_j = binary_matrix[:, w_j].sum()
                # Co-document frequency of w_i and w_j
                # Using sparse matrix dot product for speed
                co_doc_freq = binary_matrix[:, w_i].T.dot(binary_matrix[:, w_j])[0, 0]
                
                # UMass formula with +1 smoothing to avoid log(0)
                if doc_freq_j > 0:
                    topic_score += np.log((co_doc_freq + 1.0) / doc_freq_j)
                    
        coherence_score += topic_score
        
    return coherence_score / num_topics

# PARALLEL WORKER FUNCTION
def train_and_evaluate(algo_name, k, bow_mat, tfidf_mat, feature_names, binary_mat):
    """
    This is the core function that will be shipped to each CPU thread.
    """
    # Initialize the correct model based on the name
    if algo_name == 'LDA':
        model = LatentDirichletAllocation(n_components=k, random_state=42, max_iter=10)
        matrix_to_use = bow_mat # LDA must use BoW
    elif algo_name == 'NMF':
        model = NMF(n_components=k, random_state=42, init='nndsvd', max_iter=200)
        matrix_to_use = tfidf_mat # NMF uses TF-IDF
    elif algo_name == 'LSA':
        model = TruncatedSVD(n_components=k, random_state=42)
        matrix_to_use = tfidf_mat # LSA uses TF-IDF
        
    # Train the model
    model.fit(matrix_to_use)
    
    # Extract top words indices for metrics
    topic_words_indices = []
    topic_words_strings = []
    
    for topic in model.components_:
        # Get top indices
        top_indices = topic.argsort()[:-NO_TOP_WORDS - 1:-1]
        topic_words_indices.append(top_indices)
        
        # Get top actual string words for diversity
        top_strings = [feature_names[i] for i in top_indices]
        topic_words_strings.append(top_strings)
        
    # Calculate Custom Metrics
    diversity = calculate_diversity(topic_words_strings)
    coherence = calculate_umass_coherence(topic_words_indices, binary_mat)
    
    # Return a dictionary of results for this specific run
    return {
        'Algorithm': algo_name,
        'K_Topics': k,
        'Diversity': round(diversity, 3),
        'Coherence': round(coherence, 3)
    }

# EXECUTE PARALLEL GRID SEARCH 
print(f"Distributing tasks across all CPU cores (n_jobs=-1)...")
# Build the task list
tasks = [(algo, k) for algo in ALGORITHMS for k in TOPIC_RANGE]

# Run the tasks in parallel
# n_jobs=-1 tells joblib to use 100% of available CPU cores
results = Parallel(n_jobs=-1, verbose=10)(
    delayed(train_and_evaluate)(
        algo, k, bow_matrix, tfidf_matrix, feature_names, binary_bow_matrix
    ) for algo, k in tasks
)

# Convert results to a clean Pandas DataFrame for easy reading
df_results = pd.DataFrame(results)
print("\n" + "="*50)
print("              EVALUATION COMPLETED")
print("="*50)
print(df_results.sort_values(by=['Algorithm', 'K_Topics']).to_string(index=False))


# VISUALIZE THE RESULTS (Plotly)
print("\nGenerazione della dashboard interattiva...")

# Colorblind-Friendly palette
colors = {'LDA': '#0072B2', 'NMF': '#E69F00', 'LSA': '#CC79A7'}

# Initialize a figure with two side-by-side subplots
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        'UMass Topic Coherence<br><sub>(Closer to 0 is better)</sub>', 
        'Topic Diversity<br><sub>(Closer to 1.0 is better)</sub>'
    ),
    horizontal_spacing=0.1
)

# Populate the subplots with lines and markers for each algorithm
for algo in ALGORITHMS:
    subset = df_results[df_results['Algorithm'] == algo]
    
    # Graph 1: Coherence
    fig.add_trace(go.Scatter(
        x=subset['K_Topics'], 
        y=subset['Coherence'],
        mode='lines+markers',
        name=algo,
        line=dict(color=colors[algo], width=2),
        marker=dict(symbol='circle', size=8),
        legendgroup=algo,
        hovertemplate="<b>%{text}</b><br>K: %{x}<br>Coerenza: %{y:.3f}<extra></extra>",
        text=[algo] * len(subset)
    ), row=1, col=1)
    
    # Graph 2: Diversity
    fig.add_trace(go.Scatter(
        x=subset['K_Topics'], 
        y=subset['Diversity'],
        mode='lines+markers',
        name=algo,
        line=dict(color=colors[algo], width=2, dash='dash'),
        marker=dict(symbol='square', size=8),
        legendgroup=algo,
        showlegend=False, 
        hovertemplate="<b>%{text}</b><br>K: %{x}<br>Diversità: %{y:.3f}<extra></extra>",
        text=[algo] * len(subset)
    ), row=1, col=2)

fig.update_layout(
    title_text="Topic Model Evaluation Results",
    title_font_size=20,
    height=600,
    template="plotly_white",
    legend=dict(
        title="Algorithms",
        yanchor="top", y=0.99,
        xanchor="left", x=1.02
    ),
    hovermode="x unified"
)

# Axes Labels
fig.update_xaxes(title_text="Number of Topics (K)", tickmode='linear', dtick=1, row=1, col=1)
fig.update_xaxes(title_text="Number of Topics (K)", tickmode='linear', dtick=1, row=1, col=2)

fig.update_yaxes(title_text="Coherence Score", row=1, col=1)
fig.update_yaxes(title_text="Diversity Score (Proportion of unique words)", row=1, col=2)

# Show the interactive plot
fig.show()

Initializing High-Performance Parallel Topic Model Evaluator...
Distributing tasks across all CPU cores (n_jobs=-1)...


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 20 concurrent workers.
[Parallel(n_jobs=-1)]: Done   1 tasks      | elapsed:    2.2s
[Parallel(n_jobs=-1)]: Done   3 out of  24 | elapsed:    2.6s remaining:   18.3s
/home/gabriele11231/.venvs/rapids25.06_python3.12/lib/python3.12/site-packages/sklearn/decomposition/_nmf.py:1728: ConvergenceWarning: Maximum number of iterations 200 reached. Increase it to improve convergence.
  warnings.warn(
/home/gabriele11231/.venvs/rapids25.06_python3.12/lib/python3.12/site-packages/sklearn/decomposition/_nmf.py:1728: ConvergenceWarning: Maximum number of iterations 200 reached. Increase it to improve convergence.
  warnings.warn(
/home/gabriele11231/.venvs/rapids25.06_python3.12/lib/python3.12/site-packages/sklearn/decomposition/_nmf.py:1728: ConvergenceWarning: Maximum number of iterations 200 reached. Increase it to improve convergence.
  warnings.warn(
[Parallel(n_jobs=-1)]: Done   6 out of  24 | elapsed:    3.0s remaining:    9.0s
/home/gab


              EVALUATION COMPLETED
Algorithm  K_Topics  Diversity  Coherence
      LDA         3      0.600    -37.109
      LDA         4      0.575    -37.364
      LDA         5      0.540    -39.999
      LDA         6      0.600    -42.818
      LDA         7      0.600    -46.675
      LDA         8      0.613    -50.192
      LDA         9      0.578    -50.485
      LDA        10      0.590    -50.819
      LSA         3      0.767    -58.693
      LSA         4      0.725    -62.356
      LSA         5      0.600    -72.841
      LSA         6      0.567    -65.058
      LSA         7      0.471    -73.654
      LSA         8      0.438    -70.842
      LSA         9      0.456    -73.111
      LSA        10      0.410    -68.632
      NMF         3      0.867    -55.113
      NMF         4      0.875    -61.993
      NMF         5      0.880    -68.968
      NMF         6      0.867    -59.518
      NMF         7      0.800    -61.041
      NMF         8      0.787    -61.19

### Topic Extraction and Interpretation: Implementing NMF

Once the optimal number of topics has been identified through our grid search evaluation, we can proceed with the final extraction. For this phase, we isolate **Non-Negative Matrix Factorization (NMF)** as our primary algorithm. 

NMF is a powerful linear algebraic method that approximates our high-dimensional TF-IDF matrix ($V$) by factoring it into two lower-dimensional matrices.

Because NMF enforces a strict non-negativity constraint ($W \ge 0, H \ge 0$), it does not allow for subtractive relationships. Topics are constructed purely by *adding* semantic parts together. This additive property mirrors how human language naturally constructs meaning, making NMF topics highly interpretable.

Here is a methodological breakdown of the code:

* **Hyperparameter Selection:** we define `NUM_TOPICS = 6` (our $K$ dimensionality) as the empirical baseline for our specific domain, alongside `NO_TOP_WORDS = 10` to restrict our analytical focus to the most dominant features of each latent topic.
* **Deterministic Initialization (`init='nndsvd'`):** instead of initializing the matrices with random noise, we use Nonnegative Double Singular Value Decomposition (NNDSVD). This is a specialized initialization technique designed explicitly for sparse matrices (like our TF-IDF matrix). It guarantees deterministic results (the same output every run, regardless of the random seed) and drastically accelerates mathematical convergence.
* **The Extraction Logic (`model.components_`):** in scikit-learn, the `components_` attribute corresponds to the $H$ matrix (the topic-term matrix). Its shape is exactly $K \times N$ (Number of Topics $\times$ Vocabulary Size). Each row represents a topic, and each column contains the specific weight of a word within that topic.
* **Sorting and Mapping (`argsort`):** because the $H$ matrix only contains numerical weights, the `display_topics` function leverages NumPy's `argsort()` to rank the indices of these weights in ascending order. By reversing the array and slicing the top $N$ indices, we can map these numbers back to the original string tokens stored in `feature_names`, revealing the human-readable vocabulary that defines each topic.

#### Analytical Note
By simultaneously printing the exact numerical weights alongside the top words, we gain insight into the *distribution* of a topic. A topic where the top word has a weight of $2.5$ while the second has $0.3$ is heavily anchored to a single concept. Conversely, a topic with smoothly decaying weights (e.g., $1.2, 1.1, 0.9$) indicates a broader, more nuanced semantic theme.


In [ ]:
# HYPERPARAMETERS
# The number of topics to extract
NUM_TOPICS = 6
# How many top words to display to interpret the meaning of each topic
NO_TOP_WORDS = 10 

print(f"Extracting {NUM_TOPICS} hidden topics using 3 different algorithms...\n")

# ==========================================
# NON-NEGATIVE MATRIX FACTORIZATION (NMF)
# ==========================================
print("\n--- MODEL 2: NMF (TF-IDF) ---")
nmf_model = NMF(n_components=NUM_TOPICS, random_state=42, init='nndsvd')
nmf_model.fit(tfidf_matrix)

# DISPLAY FUNCTION 
def display_topics(model, feature_names, no_top_words, model_name):
    """
    Helper function to print the top features (words) for each extracted topic.
    For LSA, weights can be negative, showing words that 'repel' the topic.
    """
    for topic_idx, topic in enumerate(model.components_):
        # Sort weights in ascending order and slice the top N words
        top_words_indices = topic.argsort()[:-no_top_words - 1:-1]
        top_words = [feature_names[i] for i in top_words_indices]
        
        # Calculate rounded weights for analysis
        top_weights = [round(topic[i], 2) for i in top_words_indices]
        
        print(f"Topic {topic_idx + 1}:")
        print(" | ".join(top_words))
        print(f"Weights: {top_weights}\n")

# Retrieve the vocabulary (features) generated during vectorization
feature_names = bow_vectorizer.get_feature_names_out()

# PRINTING THE RESULTS 
print("\n" + "="*50)
print("              NMF RESULTS")
print("="*50)
display_topics(nmf_model, feature_names, NO_TOP_WORDS, "NMF")



Extracting 6 hidden topics using 3 different algorithms...


--- MODEL 2: NMF (TF-IDF) ---

              NMF RESULTS
Topic 1:
più | solo | mai | ancora | sempre | giorno | notte | mondo | qui | sentire
Weights: [2.1, 1.03, 0.85, 0.81, 0.75, 0.75, 0.74, 0.72, 0.71, 0.71]

Topic 2:
amore | amare | cuore | bello | morire | fiore | dolce | donna | cantare | grande
Weights: [3.33, 0.72, 0.59, 0.34, 0.29, 0.24, 0.22, 0.19, 0.19, 0.19]

Topic 3:
avere | soldo | stare | cazzo | chiamare | mettere | prendere | solo | amico | casa
Weights: [2.06, 0.92, 0.86, 0.73, 0.72, 0.67, 0.61, 0.61, 0.6, 0.58]

Topic 4:
sapere | dire | mai | amare | stare | pensare | solo | già | anche | cosa
Weights: [3.21, 1.49, 0.85, 0.71, 0.42, 0.4, 0.32, 0.32, 0.3, 0.3]

Topic 5:
volere | tenere | nun | cchiù | mo | aggio | pecché | ancora | pure | vita
Weights: [2.34, 0.94, 0.9, 0.86, 0.73, 0.62, 0.58, 0.46, 0.45, 0.42]

Topic 6:
andare | bene | via | così | dove | mare | stare | venire | lasciare | dire
Weights: [3.